In [6]:
import sys; sys.path.append("..")
import re, pandas as pd
from src.data import load
train, val, test = load()

pd.set_option("display.max_colwidth", 200)
print("=== 30 tweets de train ===")
for t in train["tweet"].sample(30, random_state=0): print(repr(t))
print("\n=== 30 tweets de test ===")
for t in test["tweet"].sample(30, random_state=0): print(repr(t))

=== 30 tweets de train ===
'"#MannequinChallenge انا خلوا حياتي نفسه تثبت على مود محدد عشان انا اثبت"'
'عطر غوتشي بامبو" الجديد يحاكي الطيور https://t.co/F9PRduoecU https://t.co/eGRH39QeLq"'
'"#ستبقى_حلب#ستنتصر_حلب(فدعا ربه أني مغلوب فأنتصر)والله لتنصرن ولو بعد #حين"'
'"@DROS_ASEER 📚شرح أصول السُنّة للإمام أحمد✒الشيخ سلطان العرابي📍جامع الراجحي بأبها🕗عصر الجمعة ١٨-٢-١٤٣٨🕗وبعد الم… https://t.co/DqFHkhsqQ9"'
'مااحلى شعارك ومدافعك ياقنرررزز #ارسنال'
'"RT @BassamJaara: الطائرات الروسية تحرق الاخضر واليابس والجنود الروس يسرحون ويمرحون  في شوارع حلب المحتلة ومدن اخرى لأنهم يشعرون بالامان!!"'
'إقالة لبيب السباعى من الطب الشرعى وتعيين د إحسان كميل جورجي بدلاً منه" الى مزبلة التاريخ وشكرا'
'@aldanahk @girlposts دام انه مال هاري بوتر بستعمله 💅🏼'
'"بطلبٍ من #هشام_حداد.. #رامي_عياش يحقّق حلم لين حايك https://t.co/xH63acQjcr @HichamHaddad57 @RamyAyach"'
'"@shihanh010 بوش والا اوباما😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂"'
'"في ا

In [7]:
def has_artifact(s):
    return s.startswith('"') or s.endswith('"')   

for name, df in [("train", train), ("val", val), ("test", test)]:
    a = df["tweet"].apply(has_artifact)
    print(name, "freq artefact:", round(a.mean(), 4),
          "| taux sarcasme avec:", round(df.loc[a, "sarcasm"].mean(), 3),
          "| sans:", round(df.loc[~a, "sarcasm"].mean(), 3))

train freq artefact: 0.6142 | taux sarcasme avec: 0.124 | sans: 0.251
val freq artefact: 0.6279 | taux sarcasme avec: 0.143 | sans: 0.223
test freq artefact: 0.0137 | taux sarcasme avec: 0.195 | sans: 0.275


In [8]:
import warnings; warnings.filterwarnings("ignore", category=UserWarning)
import emoji

patterns = {
    "URL": r"https?://\S+|www\.\S+",
    "mention": r"@\w+",
    "hashtag": r"#\w+",
    "rire": r"[هھ]{3,}",
    "ponct_repetee": r"[!?؟.]{2,}",
    "diacritiques": r"[\u064B-\u0652]",
    "tatweel": r"\u0640",
    "lettre_repetee": r"([\u0621-\u064A])\1{2,}",
    "latin": r"[A-Za-z]",
    "chiffres_indiens": r"[\u0660-\u0669]",
}
EMOJIS = emoji.UNICODE_EMOJI["en"]

rows = []
for name, df in [("train", train), ("val", val), ("test", test)]:
    r = {"split": name}
    for k, p in patterns.items():
        r[k] = df["tweet"].str.contains(p, regex=True).mean().round(4)
    r["emoji"] = df["tweet"].apply(lambda s: any(c in EMOJIS for c in s)).mean().round(4)
    rows.append(r)
pd.DataFrame(rows).set_index("split").T

split,train,val,test
URL,0.3809,0.3976,0.1467
mention,0.2336,0.2398,0.0077
hashtag,0.4722,0.4869,0.3290
rire,0.0117,0.0064,0.0320
ponct_repetee,0.1759,0.2000,0.2683
diacritiques,0.1186,0.1044,0.1957
tatweel,0.0455,0.0454,0.0730
lettre_repetee,0.0450,0.0438,0.1273
latin,0.5537,0.5729,0.2393
chiffres_indiens,0.0244,0.0231,0.0257


In [9]:
import re, pyarabic.araby as araby

def norm(s):
    s = re.sub(r"https?://\S+", "URL", s)
    s = re.sub(r"@\w+", "@USER", s)
    s = araby.strip_tashkeel(araby.strip_tatweel(s))
    return re.sub(r"\s+", " ", s).strip().lower()

for df in (train, val, test): df["norm"] = df["tweet"].apply(norm)

print("Doublons exacts train :", train["tweet"].duplicated().sum())
print("Doublons normalisés train :", train["norm"].duplicated().sum())

# Train ∩ test
in_test_exact = train["tweet"].isin(set(test["tweet"]))
in_test_norm  = train["norm"].isin(set(test["norm"]))
print("Train présents dans test (exact) :", in_test_exact.sum())
print("Train présents dans test (normalisé) :", in_test_norm.sum())

# Labels contradictoires dans train
g = train.groupby("norm")["sarcasm"].nunique()
contradict = g[g > 1].index
print("Groupes à labels contradictoires :", len(contradict))

Doublons exacts train : 54
Doublons normalisés train : 60
Train présents dans test (exact) : 2
Train présents dans test (normalisé) : 3
Groupes à labels contradictoires : 13


In [10]:
n_before = len(train)
train = train[~in_test_norm].copy()
print("Retirés :", n_before - len(train))

Retirés : 3


In [11]:
%load_ext autoreload
%autoreload 2
from src.data import load_clean
train_c, val_c, test_c, train_contra = load_clean()

retirés (présents dans test): 4 | (présents dans val): 72 | groupes contradictoires: 47 | doublons restants retirés: 287
tailles -> train: 10834, val: 1255, test: 3000


In [12]:
t, _, _, _ = load_clean(verbose=False)
from src.data import load
tr, _, _ = load()
import pandas as pd
from src.data import norm
tr["norm"] = tr["tweet"].apply(norm)
print(tr["norm"].value_counts().head(10))
print(train_contra.sort_values("norm")[["tweet", "sarcasm"]].head(12).to_string())

norm
ڤالنتينو كيوت وناعم. الباقيين ما أخسر فيهم ريال قديم 😳تحديدا غوتشي ! ماهذا يا بيونسيه ؟ url                                                     3
rt @user: قردوغان حليف اسرائيل يدعم تنظيم داعش في سوريا ومع ذلك لم يغضب عليه أحد ويهدده كما يفعل مع مصر التي تواجه الإرهاب بشكل عام…            3
واتس اب 0500776424 url                                                                                                                          3
rt @user: #كايت_بلانشيت ترتدي من تصميم #غوتشيخلال #أسبوع_الموضة url #ss17 url                                                                   2
طريقة تعطيل محرك البحث بينج في ويندوز 10 url                                                                                                    2
نزال الليلة سوف يدخل فيه ترامب مجروحآ. ليس لديه ما يخسره. اتوقع بيغامر وبيهاجم عائلة كلنتون خصوصآ فضيحة بيل. هيلاري ذكية سوف تتركه يدمر نفسه    2
الخارجية الامريكية القوات الأجنبية في العراق يجب أن تكون بموافقة #بغداد#تركيا #العبادي #اردوغان url                    